In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
REPLAY_JSON_PATH = PROJECT_ROOT / "replay_parser" / "output" / "parsed_match.json"

with open(REPLAY_JSON_PATH, "r") as f:
    replay_data = json.load(f)

print("Top-level keys:", list(replay_data.keys()))
for key, value in replay_data.items():
    print(f"{key}: {len(value)} entries")

Top-level keys: ['players', 'deaths', 'structures_destroyed', 'patron_events', 'mid_boss_kills', 'rift_captures', 'urn_pickups', 'rejuvenator_claims', 'stat_snapshots']
players: 12 entries
deaths: 120 entries
structures_destroyed: 19 entries
patron_events: 1 entries
mid_boss_kills: 3 entries
rift_captures: 6 entries
urn_pickups: 10 entries
rejuvenator_claims: 18 entries
stat_snapshots: 205 entries


In [2]:
# team for players
players_df = pd.DataFrame(replay_data["players"])
controller_to_team = dict(zip(players_df["controller_index"], players_df["team"]))
print(players_df)

# long format DataFrame
rows = []
for snapshot in replay_data["stat_snapshots"]:
    t = snapshot["game_time_s"]
    for p in snapshot["players"]:
        rows.append({
            "game_time_s": t,
            "controller_index": p["controller_index"],
            "team": p["team"],
            "kills": p["kills"],
            "deaths": p["deaths"],
            "assists": p["assists"],
            "hero_damage": p["hero_damage"],
            "objective_damage": p["objective_damage"],
            "gold_net_worth": p["gold_net_worth"],
        })

snapshots_long = pd.DataFrame(rows)
print(snapshots_long.shape)
snapshots_long.head(10)

    controller_index  pawn_index  team            username
0                  2        3429     2             seiryuu
1                  3        3290     2             Numlegs
2                  4        3568     2  Demui on Twitch :D
3                  5        3495     2                Tech
4                  6          77     3               Docha
5                  7          84     3              Specty
6                  8        3475     2            Seikatsu
7                  9          97     3                Anri
8                 10          98     3               Robin
9                 11          94     3     Little Boy 1945
10                12          89     2              Golova
11                13        3534     3          privativev
(2435, 9)


,game_time_s,controller_index,team,kills,deaths,assists,hero_damage,objective_damage,gold_net_worth
0,15.03125,5,2,0,0,0,0,0,600
1,15.03125,6,3,0,0,0,0,0,600
2,15.03125,10,3,0,0,0,0,0,600
3,15.03125,11,3,0,0,0,0,0,600
4,15.03125,13,3,0,0,0,0,0,600
5,30.03125,5,2,0,0,0,0,0,600
6,30.03125,6,3,0,0,0,0,0,600
7,30.03125,10,3,0,0,0,0,0,600
8,30.03125,11,3,0,0,0,0,0,600
9,30.03125,12,2,0,0,0,0,0,600


In [3]:
team_snapshots = (
    snapshots_long
    .groupby(["game_time_s", "team"], as_index=False)
    .agg({
        "kills": "sum",
        "deaths": "sum",
        "assists": "sum",
        "hero_damage": "sum",
        "objective_damage": "sum",
        "gold_net_worth": "sum",
    })
)

print(team_snapshots.shape)
print(team_snapshots.head(20))

(408, 8)
    game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
0      15.03125     2      0       0        0            0                 0   
1      15.03125     3      0       0        0            0                 0   
2      30.03125     2      0       0        0            0                 0   
3      30.03125     3      0       0        0            0                 0   
4      45.03125     2      0       0        0            0                 0   
5      45.03125     3      0       0        0            0                 0   
6      60.03125     2      0       0        0          213                 0   
7      60.03125     3      0       0        0          300                 0   
8      75.03125     2      0       0        0          887                 0   
9      75.03125     3      0       0        0          640                 0   
10     90.03125     2      0       0        0         1071                 0   
11     90.03125     3      0   

In [4]:
print(team_snapshots.tail(10))

     game_time_s  team  kills  deaths  assists  hero_damage  objective_damage  \
398  3000.046875     2     55      55      110       292164             21858   
399  3000.046875     3     55      55      106       264389             60243   
400  3015.046875     2     55      55      110       296633             21858   
401  3015.046875     3     55      55      106       267175             64569   
402  3030.046875     2     56      57      111       303760             21858   
403  3030.046875     3     57      56      116       277521             67049   
404  3045.046875     2     56      57      111       303760             21858   
405  3045.046875     3     57      56      116       277521             67049   
406  3060.046875     2     56      57      111       303760             21858   
407  3060.046875     3     57      56      116       277521             67049   

     gold_net_worth  
398          315532  
399          376216  
400          315661  
401          389154 

In [5]:
team_wide = (
    team_snapshots
    .pivot(index="game_time_s", columns="team")
)

team_wide.columns = [
    f"{stat}_team{team}"
    for stat, team in team_wide.columns
]

team_wide = team_wide.reset_index()

print(team_wide.shape)
print(team_wide.head(10))

(204, 13)
   game_time_s  kills_team2  kills_team3  deaths_team2  deaths_team3  \
0     15.03125            0            0             0             0   
1     30.03125            0            0             0             0   
2     45.03125            0            0             0             0   
3     60.03125            0            0             0             0   
4     75.03125            0            0             0             0   
5     90.03125            0            0             0             0   
6    105.03125            0            0             0             0   
7    120.03125            0            0             0             0   
8    135.03125            1            0             0             1   
9    150.03125            1            0             0             1   

   assists_team2  assists_team3  hero_damage_team2  hero_damage_team3  \
0              0              0                  0                  0   
1              0              0                  0 

In [6]:
diff_df = pd.DataFrame({
    "game_time_s": team_wide["game_time_s"],

    "kills_diff": (
        team_wide["kills_team2"] -
        team_wide["kills_team3"]
    ),

    "deaths_diff": (
        team_wide["deaths_team2"] -
        team_wide["deaths_team3"]
    ),

    "assists_diff": (
        team_wide["assists_team2"] -
        team_wide["assists_team3"]
    ),

    "hero_damage_diff": (
        team_wide["hero_damage_team2"] -
        team_wide["hero_damage_team3"]
    ),

    "objective_damage_diff": (
        team_wide["objective_damage_team2"] -
        team_wide["objective_damage_team3"]
    ),

    "net_worth_diff": (
        team_wide["gold_net_worth_team2"] -
        team_wide["gold_net_worth_team3"]
    ),
})

print(diff_df.shape)
print(diff_df.head(10))

(204, 7)
   game_time_s  kills_diff  deaths_diff  assists_diff  hero_damage_diff  \
0     15.03125           0            0             0                 0   
1     30.03125           0            0             0                 0   
2     45.03125           0            0             0                 0   
3     60.03125           0            0             0               -87   
4     75.03125           0            0             0               247   
5     90.03125           0            0             0               378   
6    105.03125           0            0             0               402   
7    120.03125           0            0             0              -326   
8    135.03125           1           -1             1               154   
9    150.03125           1           -1             1              -381   

   objective_damage_diff  net_worth_diff  
0                      0           -1800  
1                      0           -1200  
2                      0            

In [7]:
# replay event lists into DataFrames
deaths_df = pd.DataFrame(replay_data["deaths"])
structures_df = pd.DataFrame(replay_data["structures_destroyed"])
mid_boss_df = pd.DataFrame(replay_data["mid_boss_kills"])
rejuvenator_df = pd.DataFrame(replay_data["rejuvenator_claims"])

print("Deaths:")
print(deaths_df.head())

print("\nStructures:")
print(structures_df.head())

print("\nMid boss:")
print(mid_boss_df)

print("\nRejuvenator claims:")
print(rejuvenator_df.head(20))

Deaths:
   team  player_index  game_time_s  death_duration_s respawn_full_health
0     3            94   128.187500          8.062500               False
1     2            89   151.484375          8.062500               False
2     3            94   214.687500          8.046875               False
3     2            92   323.109375          8.062500               False
4     3            94   324.718750          8.062500               False

Structures:
   structure_type  team  destroyed_time_s
0  Tier2Structure     2        670.609375
1  Tier2Structure     3        784.265625
2  Tier2Structure     3        869.343750
3  Tier1Structure     2        903.500000
4  Tier2Structure     2        934.390625

Mid boss:
   killed_time_s
0    1598.609375
1    2563.921875
2    2954.109375

Rejuvenator claims:
    controller_index  team  game_time_s
0                  3     2  1605.250000
1                  2     2  1605.265625
2                 12     2  1605.265625
3                  4     2  1

In [15]:
mid_boss_with_team = []

for _, boss in mid_boss_df.iterrows():
    boss_time = boss["killed_time_s"]

    claims_after_boss = rejuvenator_df[
        (rejuvenator_df["game_time_s"] >= boss_time) &
        (rejuvenator_df["game_time_s"] <= boss_time + 30)
    ].sort_values("game_time_s")

    team_claimed = claims_after_boss.iloc[0]["team"] if len(claims_after_boss) > 0 else None

    mid_boss_with_team.append({
        "killed_time_s": boss_time,
        "team_claimed": team_claimed,
        "perspective_team": team_mapping.get(team_claimed) if team_claimed else None,
    })

mid_boss_df = pd.DataFrame(mid_boss_with_team)
print(mid_boss_df)

   killed_time_s  team_claimed perspective_team
0    1598.609375           2.0            Team0
1    2563.921875           3.0            Team1
2    2954.109375           3.0            Team1


In [16]:
FAST_RESPAWN_THRESHOLD_S = 4.0 # some extra time just in case
REJUV_WINDOW_S = 180  # 3 mins

def team_had_active_rejuv(row, rejuv_df, window_s=REJUV_WINDOW_S):
    team_claims = rejuv_df[rejuv_df["team"] == row["team"]]
    active = team_claims[
        (team_claims["game_time_s"] <= row["game_time_s"]) &
        (row["game_time_s"] <= team_claims["game_time_s"] + window_s)
    ]
    return not active.empty

deaths_df["likely_rejuv_used"] = deaths_df["death_duration_s"] < FAST_RESPAWN_THRESHOLD_S

deaths_df["rejuv_buff_active"] = deaths_df.apply(
    lambda r: team_had_active_rejuv(r, rejuvenator_df), axis=1
)

deaths_df["credit_used"] = deaths_df["likely_rejuv_used"] & deaths_df["rejuv_buff_active"]

rejuv_usage_events = deaths_df[deaths_df["credit_used"]][
    ["team", "player_index", "game_time_s", "death_duration_s"]
]
print(rejuv_usage_events)

     team  player_index  game_time_s  death_duration_s
66      2            78  1705.156250          2.984375
68      2            82  1743.828125          3.000000
71      2            92  1761.109375          2.984375
102     3            94  2729.343750          3.000000
103     3            77  2733.015625          2.984375
113     3           100  2967.468750          3.000000


In [18]:
team_wide = team_wide.rename(columns={
    col: col.replace("_team2", "_team0").replace("_team3", "_team1")
    for col in team_wide.columns
})
print(team_wide.columns.tolist())

['game_time_s', 'kills_team0', 'kills_team1', 'deaths_team0', 'deaths_team1', 'assists_team0', 'assists_team1', 'hero_damage_team0', 'hero_damage_team1', 'objective_damage_team0', 'objective_damage_team1', 'gold_net_worth_team0', 'gold_net_worth_team1']


In [20]:
structures_df["destroying_team"] = structures_df["team"].map({2: 3, 3: 2})  # whoever didn't own it destroyed it
structures_df["perspective_team"] = structures_df["destroying_team"].map(team_mapping)

# cumulative count over time per team
structures_df = structures_df.sort_values("destroyed_time_s")
structures_df["team0_cumulative"] = (structures_df["perspective_team"] == "Team0").cumsum()
structures_df["team1_cumulative"] = (structures_df["perspective_team"] == "Team1").cumsum()
print(structures_df[["destroyed_time_s", "team", "destroying_team", "team0_cumulative", "team1_cumulative"]])

    destroyed_time_s  team  destroying_team  team0_cumulative  \
0         670.609375     2                3                 0   
1         784.265625     3                2                 1   
2         869.343750     3                2                 2   
3         903.500000     2                3                 2   
4         934.390625     2                3                 2   
5         980.875000     2                3                 2   
6        1129.593750     3                2                 3   
7        1191.781250     3                2                 4   
8        1633.406250     2                3                 4   
9        1672.765625     3                2                 5   
10       1778.203125     2                3                 5   
11       2174.671875     2                3                 5   
12       2347.703125     2                3                 5   
13       2567.593750     2                3                 5   
14       2572.781250     

In [21]:
mid_boss_df = mid_boss_df.sort_values("killed_time_s")
mid_boss_df["team0_cumulative"] = (mid_boss_df["perspective_team"] == "Team0").cumsum()
mid_boss_df["team1_cumulative"] = (mid_boss_df["perspective_team"] == "Team1").cumsum()
print(mid_boss_df)

   killed_time_s  team_claimed perspective_team  team0_cumulative  \
0    1598.609375           2.0            Team0                 1   
1    2563.921875           3.0            Team1                 1   
2    2954.109375           3.0            Team1                 1   

   team1_cumulative  
0                 0  
1                 1  
2                 2  
